# AI Agents
Downloaded from Kaggle.com

!pip install pydantic==2.5.3 --q;
!pip install langchain-core==0.2.40 langchain==0.2.16 langchain-community --q ;
!pip install langchain-google-genai==1.0.10 --q;
!pip install chromadb faiss-cpu pypdf duckduckgo-search --q;

In [2]:
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ['GOOGLE_API_KEY'] = user_secrets.get_secret("google_api_key")
import requests
import json
from datetime import datetime

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import Tool
from langchain.schema import AgentAction, AgentFinish

In [4]:
tools = []
llm   = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.8)

# Simple Custom Agents

In [8]:
# SIMPLE CALCULATOR
def calculator(expr: str) -> str:
    try:
        allowed = set('0123456789+-*/()%. ')
        if not all(c in allowed for c in expr):
            return 'Error : Invalid character in expression'
        res = eval(expr)
        return f'The answer is {res}'
    except Exception as e:
        return f'Error Calculating "{expr}"'
    
    
calc_tool = Tool(
    name = 'Calculator',
    description = 'Use this for mathematical calculations with numbers, make sure input is a valid expression like "2+4/3-(42)" and "21%7" etc.',
    func = calculator, )

In [10]:
# TEXT ANALYZER
def text_analyze(text: str)-> str:
    stats = { 'word_count' : len(text.split()),        # ok for our demo
              'sentence_count' : len(text.split('.')),
              'character_count' : len(text), }
    return ''.join(f'\n\t{k} : {v}' for k, v in stats.items())

text_analysis_tool = Tool(
    name = 'Text Analyzer',
    description = 'Analyze text and provide statistics.',
    func = text_analyze, )

In [12]:
# CURRENT TIME
def get_time(timezone: str = 'UTC')-> str:
    dt = datetime.now()
    dt = dt.strftime('%d-%m-%Y %H:%M:%S')
    return f'Current timezone {timezone}, time : {dt}'

datetime_tool = Tool(
    name = 'Date and Time',
    description = 'Get the current date and time. Input can be a timezone (default in UTC)',
    func = get_time, )

In [14]:
# UNIT CONVERSION
def unit_conversion(conversion : str)-> str:
    conversion = conversion.lower().strip()

    conversions = {
            "celsius to fahrenheit": lambda c: f"{c}°C = {c * 9/5 + 32}°F",
            "fahrenheit to celsius": lambda f: f"{f}°F = {(f - 32) * 5/9}°C",
            "km to miles": lambda km: f"{km} km = {km * 0.621371} miles",
            "miles to km": lambda miles: f"{miles} miles = {miles * 1.60934} km",
            "kg to lbs": lambda kg: f"{kg} kg = {kg * 2.20462} lbs",
            "lbs to kg": lambda lbs: f"{lbs} lbs = {lbs * 0.453592} kg"
        }

    try:
        # standardizing str to : {source_val} {source_units} to {target_units}
        conv = conversion.split(' ')
        if len(conv) >= 4:
            src_value = float(conv[0])
            conv_command = " ".join(conv[1:])
            if conv_command in conversions:
                return conversion[conv_command](src_val)
        else:
            return 'use "{source_val} {source_units} to {target_units}" format for conversion'
    except Exception as e:
        return f'Error in Conversion {e}'
    
conversion_tool = Tool(
    name = 'Unit Converter',
    description = 'convert between common units. FORMAT : source_val source_units to target_units',
    func = unit_conversion, )

In [16]:
tools.append(calc_tool, text_analysis_tool, datetime_tool, conversion_tool)
for t in tools:
    print(f'{t.name} : {t.description}\n')

Calculator : Use this for mathematical calculations with numbers, make sure input is a valid expression like "2+4/3-(42)" and "21%7" etc.

Text Analyzer : Analyze text and provide statistics like word, sentence and character count.

Date and Time : Get the current date and time. Input can be a timezone (default in UTC)

Unit Converter : convert between common units. FORMAT : source_val source_units to target_units



# Creating Agents

At its core, a LangChain agent's structure is designed for dynamic decision-making and interaction with the world around it
. It's fundamentally different from a chain, which follows a predefined sequence of actions. 

Here are the major elemeents: 

+ **Language Model (LLM)**: This is the brain of the agent, responsible for understanding inputs, reasoning, and deciding the next action.
+ **Tools**: These are external resources, APIs, or functions that the agent can use to perform specific tasks, like searching the web, performing calculations, or interacting with a database.
+ **Agent Executor**: This component manages the agent's actions, executes them through the tools, and incorporates the results back into the agent's reasoning process

In [17]:
from langchain.agents import AgentType, initialize_agent

## Zero-shot React Agent (most common)

In [18]:
react_agent = initialize_agent(
    tools = tools,
    llm = llm,
    agent = AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose = True,
    max_iterations = 3,
    early_stopping_method = 'generate'
)

/tmp/ipykernel_35/903012406.py:1: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  react_agent = initialize_agent(


In [19]:
queries = [
    "What's 15% of 250?",
    "Convert 30 celsius to fahrenheit",
    "Analyze this text for word, character and sentence count: 'Machine learning is revolutionizing how we process data and make predictions.'",
    "What's the current time and what's 100 divided by 4?",
    "Convert 5 miles to km and then calculate 20% of that result"
]

In [20]:
for q in queries:
    if len(q)>150:
        q = q[:150]
    print(f'Ques : {q}')
    agent_res = react_agent.invoke(q)
    print('\n\n')

Ques : What's 15% of 250?


> Entering new AgentExecutor chain...
Action: Calculator
Action Input: 0.15 * 250
Observation: The answer is 37.5
Thought:Final Answer: 37.5

> Finished chain.



Ques : Convert 30 celsius to fahrenheit


> Entering new AgentExecutor chain...
Action: Unit Converter
Action Input: 30 celsius to fahrenheit
Observation: Error in Conversion string indices must be integers, not 'str'
Thought:Thought:The previous attempt to convert 30 celsius to fahrenheit using the Unit Converter tool resulted in an "Error in Conversion string indices must be integers, not 'str'". This error suggests an internal issue with the tool's implementation rather than an incorrect input format, as the input "30 celsius to fahrenheit" strictly followed the specified format "source_val source_units to target_units". Since I cannot directly fix the tool's internal error and no alternative tool is available for unit conversion, I cannot complete this request using the provided tools.
Final An

## Conversational Agent (with memory)

In [21]:
from langchain.agents import AgentType, initialize_agent
from langchain.memory import ConversationBufferMemory

In [22]:
memory = ConversationBufferMemory(memory_key = 'chat_history')

In [23]:
conv_agent = initialize_agent(
    tools = tools,
    llm = llm,
    memory = memory,
    agent = AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose = True
)

In [24]:
res1 = conv_agent.invoke('What do you get from 2105+2120/3102? use the calculator tool.')



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Calculator
Action Input: 2105+2120/3102
Observation: The answer is 2105.6834300451324
Thought:Do I need to use a tool? No
AI: The result of 2105 + 2120 / 3102 is approximately 2105.6834.

> Finished chain.


In [25]:
res2 = conv_agent.invoke('What is 20% of that value?')



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Calculator
Action Input: 0.20 * 2105.6834
Observation: The answer is 421.13668
Thought:Do I need to use a tool? No
AI: 20% of 2105.6834 is 421.13668.

> Finished chain.


In [26]:
for k, v in res2.items():
    print(f'{k} : {v}\n')

input : What is 20% of that value?

chat_history : Human: What do you get from 2105+2120/3102? use the calculator tool.
AI: The result of 2105 + 2120 / 3102 is approximately 2105.6834.

output : 20% of 2105.6834 is 421.13668.



## Custom Agent with Structured Tools
Instead of using pre defined tooling that takes in a single argument we can create a StructuredTool object to add flexibility and better structure to the agent tool

In [27]:
from langchain.tools import StructuredTool
from typing import Optional
from pydantic import BaseModel

In [28]:
class WeatherInp(BaseModel):
    loc: str
    units: Optional[str] = "metric"

In [29]:
def get_weather(loc: str, units: str = 'metric')-> str:
    data = {
        'bhopal' : {'temp': 38, "condition": 'sunny'},
        'berlin' : {'temp': 22, "condition": 'windy'},
        'austin' : {'temp': 19, "condition": 'cloudy'},
        'tokyo' : {'temp': 28, "condition": 'snowy'},
    }

    loc = loc.lower()
    if loc in data:
        t_data = data[loc]
        condition = t_data['condition']
        t_units = 'C' if units == 'metric' else 'F'
        temp = t_data['temp'] if units == 'metric' else t_data['temp']*9/5 + 32
        return f'Weather in {loc}: {temp}{t_units} - {condition}'
    else:
        return f'Weather for {loc} is not available.'

In [30]:
weather_tool = StructuredTool.from_function(
    func = get_weather,
    name = 'WeatherTool',
    description = 'Get Weather of a location, provide location and optionally the units (eg: metric or imperial)',
    args_schema = WeatherInp   # how to structure input (like the pydantic base model class)
)

In [31]:
s_tools = tools + [weather_tool]

In [32]:
s_agent = initialize_agent(
    tools = s_tools,
    llm = llm,
    agent = AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose = True
)

In [33]:
res = s_agent.invoke('How is the weather in Bhopal?')



> Entering new AgentExecutor chain...
Thought: The user is asking for the weather in Bhopal. I should use the WeatherTool to get this information.
Action:
```
{
  "action": "WeatherTool",
  "action_input": {
    "loc": "Bhopal"
  }
}
```

Observation: Weather in bhopal: 38C - sunny
Thought:The user asked for the weather in Bhopal. I previously used the WeatherTool and got the observation "Weather in bhopal: 38C - sunny". I can now provide this information as the final answer.
Action:
```
{
  "action": "Final Answer",
  "action_input": "The weather in Bhopal is 38°C and sunny."
}
```

> Finished chain.


In [34]:
res

{'input': 'How is the weather in Bhopal?',
 'output': 'The weather in Bhopal is 38°C and sunny.'}

# Web Search
Now we create an agent that can search the web for latest information


## Simple Search

In [56]:
import os
from langchain.agents import AgentType, initialize_agent
from langchain.tools import DuckDuckGoSearchRun

In [57]:
search_tool = DuckDuckGoSearchRun()

In [58]:
search_agent = initialize_agent(
    tools = [search_tool],
    llm = llm,
    agent = AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose = True
)

In [77]:
res = search_agent.invoke("What is the current Internet sentiment on Takopi's original sin")



> Entering new AgentExecutor chain...
Action: duckduckgo_search
Action Input: Takopi's original sin internet sentiment

/usr/local/lib/python3.11/dist-packages/langchain_community/utilities/duckduckgo_search.py:62: UserWarning: backend='api' is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(



Observation: SUMMARY Takopi's Original Sin is a dark and disturbing manga that uses a vibrant color palette to fool you. The anime has the kind of dark premise that we've been missing since all these years. The way Takopi's Original Sin handles sensitive topics is unapologetic and raw, which may cause discomfort to certain viewers. The anime adaptation for Taizan 5's Takopi's Original Sin manga was one of the more intriguing new releases of the Summer as not only was the original manga a fairly short series in comparison ... Takopi's Original Sin is one of the year's best premieres, but it's also one of its bleakest. A tragic story of good intentions gone awry unfolds…. Takopi's Original Sin (in Japanese, タコピーの原罪 or Takopii no Genzai) is a new anime that started airing on June 28, 2025. It's a type of anime called an ONA, or Original Net Animation, which means it is made especially for the internet. That's because Takopi's Original Sin comes courtesy the pen of Taizan 5, one of the mo

## Enhanced with Custom Tools

In [73]:
from langchain.tools import Tool
from langchain.agents import create_structured_chat_agent
from langchain.agents import AgentExecutor
from langchain.memory import ConversationBufferMemory
from langchain import hub

In [74]:
memory = ConversationBufferMemory()

In [62]:
class WebSearch:
    def __init__(self):
        self.engine = DuckDuckGoSearchRun()

    def seacrh(self, query : str)-> str:
        try:
            res = self.engine.invoke(query)
            return f'Search Results for query {query} : \n{res}'
        except Exception as e:
            return f'Seacrch Failed {str(e)}'

In [63]:
search = WebSearch()

In [66]:
search_tool = Tool(
    name = 'Search Web',
    description = 'Search the Internet for current information, used it in case information about current or recent events are needed.',
    func = search.seacrh
)

In [69]:
prompt_temp = hub.pull('hwchase17/structured-chat-agent')

/usr/local/lib/python3.11/dist-packages/langsmith/client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [71]:
agent = create_structured_chat_agent(
    llm = llm,
    tools = [search_tool],
    prompt = prompt_temp
)

In [81]:
agent_exec = AgentExecutor(
    agent = agent,
    tools = [search_tool],
    memory = memory,
    handle_parsing_error = True,
    verbose = True
)

In [83]:
res = agent_exec.invoke({'input':"What is the current Internet sentiment on Takopi's original sin"})



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "Search Web",
  "action_input": "Takopi's Original Sin internet sentiment"
}
```

/usr/local/lib/python3.11/dist-packages/langchain_community/utilities/duckduckgo_search.py:62: UserWarning: backend='api' is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


Search Results for query Takopi's Original Sin internet sentiment : 
Takopi's Original Sin is one of the year's best premieres, but it's also one of its bleakest. A tragic story of good intentions gone awry unfolds…. Takopi's Original Sin (in Japanese, タコピーの原罪 or Takopii no Genzai) is a new anime that started airing on June 28, 2025. It's a type of anime called an ONA, or Original Net Animation, which means it is made especially for the internet. The anime adaptation for Taizan 5's Takopi's Original Sin manga was one of the more intriguing new releases of the Summer as not only was the original manga a fairly short series in comparison ... SUMMARY Takopi's Original Sin is a dark and disturbing manga that uses a vibrant color palette to fool you. The anime has the kind of dark premise that we've been missing since all these years. The way Takopi's Original Sin handles sensitive topics is unapologetic and raw, which may cause discomfort to certain viewers. That's because Takopi's Origina

Now that's how you make simple Agents.

Now we can make the LLM run scripts programmatically, i.e., we can call APIs, make resources available and much more.

The sky is the limit!

Do Upvote and leave a comment, and keep learning 😁